<a href="https://colab.research.google.com/github/KatreenGhobrial/DataMining/blob/main/RetinexNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Clone the RetinexNet repository temporarily to extract the model weights
!git clone https://github.com/weichen582/RetinexNet.git /tmp/retinex_repo

# 2. Clone the DataMining repository temporarily to extract your specific input images
!git clone https://github.com/KatreenGhobrial/DataMining.git /tmp/datamining_repo

# 3. Create input, output, and model weights directories
!mkdir -p inputs outputs model_weights

# 4. Move the model weights to our designated folder
!mv /tmp/retinex_repo/model/* ./model_weights/

# 5. Copy all images from the DataMining 'input' folder to our local 'inputs' folder
!cp -r /tmp/datamining_repo/input/* ./inputs/

# 6. Clean up both temporary repositories to save space
!rm -rf /tmp/retinex_repo
!rm -rf /tmp/datamining_repo

# 7. Print a success message

print("Directories created, model weights loaded, and images from the DataMining repo are ready in the inputs folder!")

In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np
from PIL import Image
import glob

# --- Helper Functions ---
def load_images(file):
    im = Image.open(file)
    return np.array(im, dtype="float32") / 255.0

def save_images(filepath, result_1):
    result_1 = np.squeeze(result_1)
    im = Image.fromarray(np.clip(result_1 * 255.0, 0, 255.0).astype('uint8'))
    im.save(filepath, 'png')

# --- Network Architecture ---
def concat(layers):
    return tf.concat(layers, axis=3)

def DecomNet(input_im, layer_num, channel=64, kernel_size=3):
    input_max = tf.reduce_max(input_im, axis=3, keepdims=True)
    input_im = concat([input_max, input_im])
    with tf.variable_scope('DecomNet', reuse=tf.AUTO_REUSE):
        conv = tf.compat.v1.layers.conv2d(input_im, channel, kernel_size * 3, padding='same', activation=None, name="shallow_feature_extraction")
        for idx in range(layer_num):
            conv = tf.compat.v1.layers.conv2d(conv, channel, kernel_size, padding='same', activation=tf.nn.relu, name='activated_layer_%d' % idx)
        conv = tf.compat.v1.layers.conv2d(conv, 4, kernel_size, padding='same', activation=None, name='recon_layer')
    R = tf.sigmoid(conv[:,:,:,0:3])
    L = tf.sigmoid(conv[:,:,:,3:4])
    return R, L

def RelightNet(input_L, input_R, channel=64, kernel_size=3):
    input_im = concat([input_R, input_L])
    with tf.variable_scope('RelightNet'):
        conv0 = tf.compat.v1.layers.conv2d(input_im, channel, kernel_size, padding='same', activation=None)
        conv1 = tf.compat.v1.layers.conv2d(conv0, channel, kernel_size, strides=2, padding='same', activation=tf.nn.relu)
        conv2 = tf.compat.v1.layers.conv2d(conv1, channel, kernel_size, strides=2, padding='same', activation=tf.nn.relu)
        conv3 = tf.compat.v1.layers.conv2d(conv2, channel, kernel_size, strides=2, padding='same', activation=tf.nn.relu)
        up1 = tf.image.resize_nearest_neighbor(conv3, (tf.shape(conv2)[1], tf.shape(conv2)[2]))
        deconv1 = tf.compat.v1.layers.conv2d(up1, channel, kernel_size, padding='same', activation=tf.nn.relu) + conv2
        up2 = tf.image.resize_nearest_neighbor(deconv1, (tf.shape(conv1)[1], tf.shape(conv1)[2]))
        deconv2 = tf.compat.v1.layers.conv2d(up2, channel, kernel_size, padding='same', activation=tf.nn.relu) + conv1
        up3 = tf.image.resize_nearest_neighbor(deconv2, (tf.shape(conv0)[1], tf.shape(conv0)[2]))
        deconv3 = tf.compat.v1.layers.conv2d(up3, channel, kernel_size, padding='same', activation=tf.nn.relu) + conv0
        deconv1_resize = tf.image.resize_nearest_neighbor(deconv1, (tf.shape(deconv3)[1], tf.shape(deconv3)[2]))
        deconv2_resize = tf.image.resize_nearest_neighbor(deconv2, (tf.shape(deconv3)[1], tf.shape(deconv3)[2]))
        feature_gather = concat([deconv1_resize, deconv2_resize, deconv3])
        feature_fusion = tf.compat.v1.layers.conv2d(feature_gather, channel, 1, padding='same', activation=None)
        output = tf.compat.v1.layers.conv2d(feature_fusion, 1, 3, padding='same', activation=None)
    return output

class RetinexModel:
    def __init__(self, sess):
        self.sess = sess
        self.input_low = tf.placeholder(tf.float32, [None, None, None, 3], name='input_low')

        [R_low, I_low] = DecomNet(self.input_low, layer_num=5)
        I_delta = RelightNet(I_low, R_low)
        I_delta_3 = concat([I_delta, I_delta, I_delta])

        # --- חשיפת המשתנים החוצה לצורך ניקוי הרעשים ---
        self.output_R_low = R_low
        self.output_I_delta_3 = I_delta_3

        self.var_Decom = [var for var in tf.trainable_variables() if 'DecomNet' in var.name]
        self.var_Relight = [var for var in tf.trainable_variables() if 'RelightNet' in var.name]

        self.saver_Decom = tf.train.Saver(var_list=self.var_Decom)
        self.saver_Relight = tf.train.Saver(var_list=self.var_Relight)

    def load_weights(self, model_dir):
        ckpt_decom = tf.train.get_checkpoint_state(os.path.join(model_dir, 'Decom'))
        ckpt_relight = tf.train.get_checkpoint_state(os.path.join(model_dir, 'Relight'))

        if ckpt_decom and ckpt_relight:
            self.saver_Decom.restore(self.sess, tf.train.latest_checkpoint(os.path.join(model_dir, 'Decom')))
            self.saver_Relight.restore(self.sess, tf.train.latest_checkpoint(os.path.join(model_dir, 'Relight')))
            return True
        return False

In [ ]:
import cv2
import glob
import os
import numpy as np

input_dir = './inputs'

# Find all images in the folder (filtering out already augmented ones to avoid an infinite loop)
image_paths = glob.glob(os.path.join(input_dir, '*.*'))
original_images = [p for p in image_paths if '_aug_' not in p]

if not original_images:
    print("No original images found in the directory.")
else:
    print(f"Found {len(original_images)} images. Generating variations...")

    for img_path in original_images:
        filename = os.path.basename(img_path)
        name, ext = os.path.splitext(filename)

        # Read the image as a matrix of pixels
        img = cv2.imread(img_path)
        if img is None:
            continue

        # Manipulation 1: Horizontal Flip (Mirror)
        img_flipped = cv2.flip(img, 1)
        save_path_flipped = os.path.join(input_dir, f"{name}_aug_flipped{ext}")
        cv2.imwrite(save_path_flipped, img_flipped)
        print(f"Created flipped image: {save_path_flipped}")

        # Manipulation 2: Add Random "Noise"
        # Generate a matrix of random noise and add it to the existing pixels
        noise = np.random.normal(0, 15, img.shape).astype(np.float32)
        img_noisy = cv2.add(img.astype(np.float32), noise)
        img_noisy = np.clip(img_noisy, 0, 255).astype(np.uint8) # Ensure pixels stay within the valid 0-255 range
        save_path_noisy = os.path.join(input_dir, f"{name}_aug_noisy{ext}")
        cv2.imwrite(save_path_noisy, img_noisy)
        print(f"Created noisy image: {save_path_noisy}")

        # Manipulation 3: Extreme Darkening (Multiply pixel values by 0.5)
        img_darker = cv2.convertScaleAbs(img, alpha=0.5, beta=0)
        save_path_darker = os.path.join(input_dir, f"{name}_aug_darker{ext}")
        cv2.imwrite(save_path_darker, img_darker)
        print(f"Created darker image: {save_path_darker}")

    print("\nDone! The new images are waiting in the inputs folder.")

In [ ]:
import cv2
import glob
import os
import numpy as np

input_dir = './inputs'
output_dir = './outputs'
model_dir = './model_weights'

# מציאת כל התמונות בתיקיית הקלט
test_low_data_names = glob.glob(os.path.join(input_dir, '*.*'))

if not test_low_data_names:
    print(f"No images found in {input_dir}.")
else:
    print(f"Found {len(test_low_data_names)} images. Starting processing WITH Denoising...")

    tf.reset_default_graph()
    with tf.Session() as sess:
        model = RetinexModel(sess)
        sess.run(tf.global_variables_initializer())

        if model.load_weights(model_dir):
            print("Model weights loaded! Enhancing and denoising images...")
            for img_path in test_low_data_names:
                filename = os.path.basename(img_path)
                name, ext = os.path.splitext(filename)

                # טעינת התמונה המקורית
                img = load_images(img_path)
                img_input = np.expand_dims(img, axis=0)

                # שלב א': שולפים את R ו-I מהמודל בנפרד! (לא מחפשים יותר את output_S)
                [R_out, I_out] = sess.run([model.output_R_low, model.output_I_delta_3], feed_dict={model.input_low: img_input})

                R_sq = np.squeeze(R_out)
                I_sq = np.squeeze(I_out)

                # שלב ב': תהליך Denoising על רכיב ה-R באמצעות OpenCV
                # 1. המרה לפורמט של תמונה רגילה (0-255)
                R_uint8 = np.clip(R_sq * 255.0, 0, 255).astype(np.uint8)

                # 2. הפעלת מנקה הרעשים הקלאסי (Fast Non-Local Means)
                R_denoised_uint8 = cv2.fastNlMeansDenoisingColored(R_uint8, None, 15, 15, 7, 21)

                # 3. המרה חזרה למספרים עשרוניים (0.0 - 1.0) כדי שהמתמטיקה תעבוד
                R_denoised = R_denoised_uint8.astype(np.float32) / 255.0

                # שלב ג': הבנייה מחדש - הכפלת ה-R הנקי בתאורה החדשה I
                S_final = R_denoised * I_sq

                # שמירה לתיקיית הפלט
                save_path = os.path.join(output_dir, f"{name}_improved.png")
                save_images(save_path, S_final)
                print(f"Successfully saved (Denoised!): {save_path}")
        else:
            print("Error! Could not load model weights.")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

test_images = glob.glob(os.path.join(input_dir, '*.*'))

for img_path in test_images:
    filename = os.path.basename(img_path)
    name, _ = os.path.splitext(filename)
    output_path = os.path.join(output_dir, f"{name}_improved.png")

    if os.path.exists(output_path):
        fig, axes = plt.subplots(1, 2, figsize=(15, 7))

        # Display original image
        axes[0].imshow(mpimg.imread(img_path))
        axes[0].set_title(f"Original: {filename}")
        axes[0].axis('off')

        # Display enhanced image
        axes[1].imshow(mpimg.imread(output_path))
        axes[1].set_title(f"Enhanced: {name}_improved.png")
        axes[1].axis('off')

        plt.show()

In [ ]:
import cv2
import glob
import os
import numpy as np

def calculate_loe(original_img, enhanced_img, resize_dim=50):
    """
    Calculates the Lightness Order Error (LOE) between original and enhanced images.
    Lower LOE means better lightness order preservation (more natural look).
    """
    # Convert images to float32
    orig_float = original_img.astype(np.float32)
    enh_float = enhanced_img.astype(np.float32)

    # Get Lightness (maximum value among RGB channels)
    L_orig = np.max(orig_float, axis=2)
    L_enh = np.max(enh_float, axis=2)

    # Downsample images to speed up the heavy O(N^2) computation
    L_orig_small = cv2.resize(L_orig, (resize_dim, resize_dim))
    L_enh_small = cv2.resize(L_enh, (resize_dim, resize_dim))

    # Flatten the matrices to 1D arrays
    orig_flat = L_orig_small.flatten()
    enh_flat = L_enh_small.flatten()

    # Compare each pixel with all other pixels (Broadcasting)
    # This creates a boolean matrix where True means Pixel A >= Pixel B
    U_orig = orig_flat[:, np.newaxis] >= orig_flat[np.newaxis, :]
    U_enh = enh_flat[:, np.newaxis] >= enh_flat[np.newaxis, :]

    # Calculate the absolute difference between the order matrices
    # 0 means order was preserved, 1 means order was flipped
    order_diff = np.abs(U_orig.astype(int) - U_enh.astype(int))

    # LOE is the mean of the sum of order errors for each pixel
    loe_score = np.mean(np.sum(order_diff, axis=1))

    return loe_score

# --- Main Execution ---
input_dir = './inputs'
output_dir = './outputs'

# Find all input images
test_images = glob.glob(os.path.join(input_dir, '*.*'))

print("=== Lightness Order Error (LOE) Evaluation ===")
print("NOTE: Lower LOE score is better (indicates natural lighting preservation).\n")

for img_path in test_images:
    filename = os.path.basename(img_path)
    name, ext = os.path.splitext(filename)
    output_path = os.path.join(output_dir, f"{name}_improved.png")

    if os.path.exists(output_path):
        # Load the before and after images
        img_in = cv2.imread(img_path)
        img_out = cv2.imread(output_path)

        # Calculate LOE
        loe_value = calculate_loe(img_in, img_out)

        print(f"Image: {filename}")
        print(f"  > LOE Score: {loe_value:.2f}")
        print("-" * 30)